# 4.1. Задача Коши

Реализация: `lab4/ode_ivp.py` (Эйлер, Рунге–Кутта 4, Адамс 4). Данные — из `data/lab4/4_1/`. Правая часть задаётся callable; размерность системы не зашита в код.

ОДУ: $y'' + y'\operatorname{tg} x - y\cos^2 x = 0$, $y(0)=2$, $y'(0)=0$, $x\in[0,1]$, $h=0.1$.

Точное решение (согласованное с ОДУ и НУ): $y=e^{\sin x}+e^{-\sin x}$.

In [1]:
import sys
import math
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "lab4"

from lab4.dataio import read_interval, read_scalar, read_vector, read_bc
from lab4.helpers import runge_romberg, fmt_row
from lab4.ode_ivp import euler, rk4, adams4, second_order_to_system
from lab4.bvp import shooting, finite_difference, linear_rhs_from_coefs

a, b = read_interval(DATA / "4_1" / "interval.txt")
h = read_scalar(DATA / "4_1" / "h.txt")
y0 = read_vector(DATA / "4_1" / "y0.txt")
print(f"отрезок [{a}, {b}], h = {h}, y0 = {y0}")

отрезок [0.0, 1.0], h = 0.1, y0 = [2.0, 0.0]


## Правая часть и точное решение

In [2]:
def rhs(x, y, yp):
    return -yp * math.tan(x) + y * (math.cos(x) ** 2)

def exact(x):
    return math.exp(math.sin(x)) + math.exp(-math.sin(x))

f = second_order_to_system(rhs)

## Эйлер, RK4, Адамс

In [3]:
xs_e, ys_e = euler(f, a, y0, h, b)
xs_r, ys_r = rk4(f, a, y0, h, b)
xs_a, ys_a = adams4(f, a, y0, h, b)

print(f"{'x':>8} {'Euler':>14} {'RK4':>14} {'Adams':>14} {'exact':>14}")
for i, x in enumerate(xs_e):
    ye = exact(x)
    print(fmt_row([x, ys_e[i][0], ys_r[i][0], ys_a[i][0], ye]))

def max_err(xs, ys):
    return max(abs(ys[i][0] - exact(xs[i])) for i in range(len(xs)))

print("max |y - exact|: Euler =", max_err(xs_e, ys_e))
print("max |y - exact|: RK4   =", max_err(xs_r, ys_r))
print("max |y - exact|: Adams =", max_err(xs_a, ys_a))

       x          Euler            RK4          Adams          exact
             0              2              2              2              2
           0.1              2       2.009975       2.009975       2.009975
           0.2           2.02      2.0395996      2.0395996      2.0395995
           0.3         2.0596      2.0879697      2.0879697      2.0879696
           0.4         2.1178      2.1535729      2.1535745      2.1535728
           0.5      2.1929969      2.2342855      2.2342845      2.2342853
           0.6       2.282981      2.3273825      2.3273665      2.3273822
           0.7      2.3849387        2.42957       2.429521      2.4295697
           0.8      2.4954722      2.5370498      2.5369438      2.5370495
           0.9      2.6106471      2.6456257      2.6454354      2.6456254
             1      2.7260762      2.7508531      2.7505526      2.7508528
max |y - exact|: Euler = 0.04463101898209665
max |y - exact|: RK4   = 3.1468522987410097e-07
max |y - exac

## Рунге–Ромберг (шаги h и 2h)

In [4]:
def rr_report(name, solver, p):
    xs1, ys1 = solver(f, a, y0, h, b)
    xs2, ys2 = solver(f, a, y0, 2 * h, b)
    yf = [row[0] for row in ys1]
    yc = [row[0] for row in ys2]
    refined, R = runge_romberg(yf, yc, p)
    print(f"=== {name}, p={p} ===")
    print(f"{'x':>8} {'y(h)':>14} {'y(2h)':>14} {'R':>14} {'y_RR':>14} {'exact':>14}")
    for i, x in enumerate(xs2):
        print(fmt_row([x, yf[2 * i], yc[i], R[i], refined[i], exact(x)]))
    print("max |R| =", max(abs(r) for r in R))
    print("max |y_RR - exact| =", max(abs(refined[i] - exact(xs2[i])) for i in range(len(xs2))))
    print()

rr_report("Euler", euler, 1)
rr_report("RK4", rk4, 4)
rr_report("Adams", adams4, 4)

=== Euler, p=1 ===
       x           y(h)          y(2h)              R           y_RR          exact
             0              2              2              0              2              2
           0.2           2.02              2           0.02           2.04      2.0395995
           0.4         2.1178           2.08    0.037799977         2.1556      2.1535728
           0.6       2.282981      2.2335991     0.04938195       2.332363      2.3273822
           0.8      2.4954722       2.444793    0.050679167      2.5461514      2.5370495
             1      2.7260762      2.6879491    0.038127113      2.7642033      2.7508528
max |R| = 0.05067916668777439
max |y_RR - exact| = 0.013350536750888153

=== RK4, p=4 ===
       x           y(h)          y(2h)              R           y_RR          exact
             0              2              2              0              2              2
           0.2      2.0395996      2.0396013 -1.1833188e-07      2.0395994      2.0395995
   